# FEN Piura V2 | Proceso de optimización cuántico-híbrida

**Caso:** distribución de ayuda humanitaria a albergues durante escenarios del Fenómeno El Niño (FEN).  
**Plataforma:** Python / HiGHS / Qiskit / IBM Quantum opcional.  
**Estado:** V2 experimental; **no apto para despacho real**. Los puntos y daños del caso son ficticios.

### ¿Qué hace este notebook?
1. Construye el caso de 8 puntos heredado y escenarios sintéticos de 10, 25 y 50.
2. Genera rutas candidatas **factibles** con capacidad, ventanas horarias, vehículos especiales y viajes de ida y vuelta.
3. Resuelve la selección global de rutas mediante un **MILP/HiGHS** (baseline).
4. Crea un subproblema QUBO de selección de rutas (10 bits como máximo) y lo compara contra enumeración y azar.
5. Entrena y muestrea QAOA ideal en la computadora local; opcionalmente ejecuta el mismo circuito en una **QPU IBM**.
6. Simula replanificación **antes de despachar** por nuevos factores de desvío.
7. Exporta planes y tablas para el dashboard de la competencia.

**Límites explícitos:** aún no contiene red vial real OSM, despacho en tiempo real, PyVRP/OR-Tools ni entrenamiento variacional en hardware. No confundir óptimo del conjunto de rutas generado con el óptimo del VRPTW original cuando se usa un pool de rutas limitado.

## 0. Instalar dependencias
En Colab ejecuta esta celda. `qiskit` y `qiskit-ibm-runtime` se instalan solo si utilizarás la QPU opcional.

In [ ]:
%pip -q install "numpy>=1.26,<3" "scipy>=1.12,<2" "pandas>=2,<3" "matplotlib>=3.8,<4"

## 1. Núcleo autocontenido
La siguiente celda escribe `fen_v2.py` en la carpeta de trabajo; permite usar **este notebook por sí solo** en Colab. En el ZIP del equipo, el mismo módulo se mantiene en `src/fen_v2.py` para control de versiones.

In [ ]:
%%writefile fen_v2.py
"""FEN Piura V2: reproducible prototype (synthetic input, classical benchmark,
route-based small QUBO, local ideal QAOA, optional IBM adapter).

NOT for real emergency dispatch. Scenario locations/demands/speeds are illustrative.
"""
from __future__ import annotations
import itertools
import json
import math
import time
from dataclasses import dataclass, replace, asdict
from pathlib import Path
from typing import Optional
import numpy as np
from scipy.optimize import Bounds, LinearConstraint, milp, minimize
from scipy.sparse import lil_matrix

SEED = 20261024
DEPOT = ("D0", -5.1945, -80.6328)
BASE_RAW = [
    ("P1", "Los Algarrobos", "Catacaos", -5.2670,-80.6760,  4,"06:30","08:00",3,False),
    ("P2", "Nueva Esperanza", "Cura Mori",-5.3230,-80.6640, 2,"07:00","09:30",3,True),
    ("P3", "El Porvenir", "El Tallán",    -5.4100,-80.6830, 1,"07:00","10:30",2,True),
    ("P4", "Santa Rosa", "La Arena",      -5.3440,-80.7110, 3,"08:00","12:00",2,False),
    ("P5", "Las Palmeras", "La Unión",    -5.3990,-80.7420, 3,"08:00","13:00",2,False),
    ("P6", "San Martín", "Vice",           -5.4180,-80.7790, 2,"09:00","15:00",1,False),
    ("P7", "Los Ceibos", "Bernal",         -5.4600,-80.7430, 1,"07:00","07:45",3,False),
    ("P8", "Puerto Nuevo", "Sechura",     -5.5580,-80.8210, 3,"10:00","16:00",1,False),
]
# Original notebook computes demand from rounded 2017 district populations;
# the illustrative demands above reproduce the source logic (1..4 pallets).


def minutes(t: str)->int:
    h,m=map(int,t.split(':'));return 60*h+m

def clock(t:float)->str:
    t=int(round(t));return f'{t//60:02d}:{t%60:02d}'

@dataclass(frozen=True)
class Stop:
    id:str; name:str; region:str; lat:float; lon:float; demand:int; start:int; end:int; priority:int; flooded:bool=False; source:str='synthetic'

@dataclass(frozen=True)
class Vehicle:
    id:str; name:str; capacity:int; speed_kmh:float; offroad:bool

@dataclass(frozen=True)
class TripSlot:
    id:str; vehicle:str; index:int; start:int; end:int

@dataclass
class Scenario:
    name:str; stops:dict[str,Stop]; vehicles:dict[str,Vehicle]; slots:list[TripSlot];
    detours:dict[tuple[str,str],float]; max_stops:int=3; start:int=360; end:int=1080;
    route_factor:float=1.3; priority_weight:float=1.0; wait_weight:float=0.05

@dataclass(frozen=True)
class Route:
    id:str; slot:str; vehicle:str; points:tuple[str,...]; cost:float;
    drive:float; wait:float; return_at:float; load:int


def make_scenario(n:int=8, seed:int=SEED, detour_override:Optional[dict]=None)->Scenario:
    """8 replicates prior fictional PoC; 10/25/50 extend with synthetic jitter.
    Multi-trip slots are conceptual bounded shifts, not mid-route live replanning.
    """
    if n not in (8,10,25,50):raise ValueError('Tamaños previstos: 8, 10, 25 o 50')
    stops={}
    for ident,name,region,lat,lon,d,a,b,p,flood in BASE_RAW:
        stops[ident]=Stop(ident,name,region,lat,lon,d,minutes(a),minutes(b),p,flood)
    rng=np.random.default_rng(seed)
    for k in range(9,n+1):
        anchor=BASE_RAW[(k-9)%8]
        _,_,reg,lat,lon,*_=anchor
        # These are SYNTHETIC samples, not validated aid shelters or real roads.
        st=Stop(f'S{k:02d}',f'Punto sintético {k}',reg,
                lat+float(rng.uniform(-0.013,0.013)),lon+float(rng.uniform(-0.013,0.013)),
                int(rng.choice([1,1,2])),minutes('06:00'),minutes('18:00'),
                int(rng.integers(1,4)),False)
        stops[st.id]=st
    base=[Vehicle('V1','Camión 6 t',10,35,False), Vehicle('V2','Camioneta 4x4',4,50,True),
          Vehicle('V3','Camión 4x4',7,40,True)]
    if n==8:
        v=base; trips=1
    elif n==10:
        v=base+[Vehicle('V4','Camión 4x4 B',8,42,True)];trips=1
    elif n==25:
        v=base+[Vehicle('V4','Camión 4x4 B',8,42,True),Vehicle('V5','Camión 6 t B',10,38,False)];trips=2
    else:
        v=base+[Vehicle('V4','Camión 4x4 B',8,42,True),Vehicle('V5','Camión 6 t B',10,38,False),
                Vehicle('V6','Camioneta 4x4 B',5,50,True),Vehicle('V7','Camión 4x4 C',8,40,True),
                Vehicle('V8','Camión 6 t C',10,38,False)]; trips=3
    slots=[TripSlot(f'{veh.id}:T{j+1}',veh.id,j,360+j*int(720/trips),360+(j+1)*int(720/trips))
           for veh in v for j in range(trips)]
    detours={tuple(sorted(k)):val for k,val in {('P5','P6'):3.0,('D0','P8'):1.4}.items()}
    if detour_override:
        for key,mult in detour_override.items():detours[tuple(sorted(key))]=float(mult)
    return Scenario(f'Bajo Piura · {n} puntos · datos ficticios',stops,{x.id:x for x in v},slots,detours)


def coords(s:Scenario,p:str)->tuple[float,float]:
    return (DEPOT[1],DEPOT[2]) if p=='D0' else (s.stops[p].lat,s.stops[p].lon)


def km(s:Scenario,a:str,b:str)->float:
    if a==b:return 0.0
    lat1,lon1=map(math.radians,coords(s,a));lat2,lon2=map(math.radians,coords(s,b))
    dlat=lat2-lat1;dlon=lon2-lon1
    h=math.sin(dlat/2)**2+math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return 2*6371*math.asin(min(1,math.sqrt(h)))*s.route_factor


def travel(s:Scenario,a:str,b:str,veh:str)->float:
    return km(s,a,b)*60/s.vehicles[veh].speed_kmh*s.detours.get(tuple(sorted((a,b))),1.)


def simulate(s:Scenario,slot:TripSlot,points:tuple[str,...]|list[str])->Optional[Route]:
    """Returns a feasible candidate; hard windows interpreted as start of service <= closing."""
    if len(points)>s.max_stops or len(set(points))!=len(points):return None
    v=s.vehicles[slot.vehicle];load=sum(s.stops[p].demand for p in points)
    if load>v.capacity:return None
    if any(s.stops[p].flooded and not v.offroad for p in points):return None
    t=float(slot.start);dr=0.;waiting=0.;cost_priority=0.;prev='D0'
    for p in points:
        st=s.stops[p];dt=travel(s,prev,p,v.id);t+=dt;dr+=dt
        waiting+=max(0.,st.start-t)
        t=max(t,st.start)
        if t>st.end+1e-7:return None
        cost_priority += s.priority_weight*(st.priority-1)*max(0,t-s.start)/60
        t+=15+5*st.demand;prev=p
    dt=travel(s,prev,'D0',v.id) if points else 0.;t+=dt;dr+=dt
    if t>slot.end+1e-7:return None
    # cost expressed in minutes-equivalent; service minutes constant if all served.
    return Route(f'{slot.id}|{"-".join(points)}',slot.id,slot.vehicle,tuple(points),
                 dr+s.wait_weight*waiting+cost_priority,dr,waiting,t,load)


def route_pool(s:Scenario, seed:int=SEED)->list[Route]:
    """Exhaustive route pool at 8; deterministic geographically diversified pools otherwise.
    The ≥10-point benchmarks are BEST-KNOWN for this sampled column pool, not global optima.
    """
    ids=list(s.stops);ret={}
    rng=np.random.default_rng(seed)
    for ix,slot in enumerate(s.slots):
        eligible=[p for p in ids if not(s.stops[p].flooded and not s.vehicles[slot.vehicle].offroad)]
        if len(ids)<=10:
            focused=eligible
        else:
            # Each slot gets a geographical focus, rotating across anchors, plus a random complement.
            anchors=list(s.stops)
            center=anchors[(ix*7+ix//len(s.vehicles))%len(anchors)]
            nearest=sorted(eligible,key=lambda p:km(s,center,p))[:11]
            extra=list(rng.choice(eligible,size=min(4,len(eligible)),replace=False))
            focused=list(dict.fromkeys(nearest+extra))
        for p in eligible: # Singletons ensure that a distant point isn't absent from the pool.
            r=simulate(s,slot,(p,))
            if r:ret[r.id]=r
        for size in range(2,s.max_stops+1):
            for group in itertools.combinations(focused,size):
                for perm in itertools.permutations(group):
                    r=simulate(s,slot,perm)
                    if r:ret[r.id]=r
    return list(ret.values())


def validate(s:Scenario,chosen:list[Route],required:Optional[set[str]]=None)->dict:
    required=set(s.stops) if required is None else set(required)
    served=list(itertools.chain.from_iterable(r.points for r in chosen))
    slots=[r.slot for r in chosen]
    tests={
        'cobertura_exacta':len(served)==len(required) and set(served)==required,
        'sin_repetidos':len(served)==len(set(served)),
        'slot_unico':len(slots)==len(set(slots)),
        'capacidad_ventanas_acceso_y_retorno':all(next((simulate(s,sl,r.points) is not None for sl in s.slots if sl.id==r.slot),False) for r in chosen),
    }
    return {'ok':all(tests.values()),'checks':tests,'atendidos':len(set(served)),'solicitudes':len(required)}


def solve_master(s:Scenario,routes:list[Route],time_limit:float=30.,required:Optional[set[str]]=None)->dict:
    """Set-partitioning MILP: each stop exactly once, each vehicle-trip slot <= 1.
    Output optimal only if HiGHS certifies status 0; otherwise incumbent feasible.
    """
    t0=time.perf_counter()
    required=set(s.stops) if required is None else set(required)
    rr=[r for r in routes if set(r.points).issubset(required)]
    stop_ids=sorted(required);slot_ids=sorted({r.slot for r in rr})
    if not rr or any(not any(p in r.points for r in rr) for p in stop_ids):
        return dict(status='infeasible_pool',ok=False,routes=[],elapsed=time.perf_counter()-t0)
    rows=len(stop_ids)+len(slot_ids)
    A=lil_matrix((rows,len(rr)),dtype=np.float64)
    si={p:i for i,p in enumerate(stop_ids)};sl={sid:len(stop_ids)+j for j,sid in enumerate(slot_ids)}
    for j,r in enumerate(rr):
        for p in r.points:A[si[p],j]=1
        A[sl[r.slot],j]=1
    lb=np.r_[np.ones(len(stop_ids)),np.zeros(len(slot_ids))]
    ub=np.ones(rows)
    c=np.asarray([r.cost for r in rr],float)
    out=milp(c,integrality=np.ones(len(rr),int),
             bounds=Bounds(np.zeros(len(rr)),np.ones(len(rr))),
             constraints=LinearConstraint(A.tocsr(),lb,ub),
             options={'time_limit':time_limit,'mip_rel_gap':0.001})
    chosen=[r for j,r in enumerate(rr) if out.x is not None and out.x[j]>.5]
    checks=validate(s,chosen,required)
    return dict(status=out.message,status_code=int(out.status),optimal=out.status==0,ok=checks['ok'],
                obj=sum(r.cost for r in chosen) if checks['ok'] else None,
                bound=getattr(out,'mip_dual_bound',None),gap=getattr(out,'mip_gap',None),
                elapsed=time.perf_counter()-t0, pool_size=len(rr),routes=chosen,validation=checks)


def summary(s:Scenario,result:dict)->dict:
    out={k:result.get(k) for k in ('status','optimal','ok','obj','bound','gap','elapsed','pool_size')}
    chosen=result.get('routes',[])
    out.update(n=len(s.stops),vehicles=len(s.vehicles),trip_slots=len(s.slots),routes_used=len(chosen),
               km_est=sum(km(s,a,b) for r in chosen for a,b in zip(('D0',)+r.points,r.points+('D0',))),
               travel_minutes=sum(r.drive for r in chosen),
               priority_served_pct=100*sum(s.stops[p].priority==3 for r in chosen for p in r.points)/max(1,sum(x.priority==3 for x in s.stops.values())))
    return out


def select_neighborhood(s:Scenario,base:dict,pool:list[Route],max_variables:int=10)->dict:
    """Construct a small, feasible, connected incumbent neighborhood with paired covers.
    Keeps alternatives on two already-used slots and leaves all other trips fixed.
    """
    selected=sorted(base['routes'],key=lambda r:r.slot)
    if len(selected)<2:raise ValueError('Necesita 2 rutas usadas como mínimo')
    best_choice=None
    for left,right in itertools.combinations(selected,2):
        req=set(left.points+right.points)
        lefts=[r for r in pool if r.slot==left.slot and set(r.points).issubset(req)]
        rights=[r for r in pool if r.slot==right.slot and set(r.points).issubset(req)]
        keys={frozenset(r.points):[] for r in rights}
        for r in rights:keys[frozenset(r.points)].append(r)
        pairs=[]
        for r in lefts:
            complement=frozenset(req-set(r.points))
            for other in keys.get(complement,[]):
                pairs.append((r,other))
        # Favor pairs with several substitutions, especially nonempty routes.
        score=(min(len(pairs),40),len(req),-sum(r.cost for r in (left,right)))
        if best_choice is None or score>best_choice[0]:
            best_choice=(score,req,(left,right),pairs)
    _,req,incumbent,pairs=best_choice
    options=list(incumbent)
    for a,b in sorted(pairs,key=lambda pair:pair[0].cost+pair[1].cost):
        fresh=[r for r in (a,b) if r.id not in {z.id for z in options}]
        if len(options)+len(fresh)<=max_variables:options.extend(fresh)
    return {'routes':options,'required':req,'slots':{r.slot for r in incumbent},'incumbent':list(incumbent),
            'complete_pairs_available':len(pairs)}


def make_qubo(neigh:dict,penalty_scale:float=3.)->dict:
    routes=neigh['routes'];req=neigh['required'];n=len(routes)
    # Energy E(x)=const + diagonal*x + pairwise*x_i*x_j, integer coverage exact.
    costs=np.array([r.cost for r in routes],float)
    pen=penalty_scale*max(1.,sum(abs(costs)))
    lin=costs.copy();pair=np.zeros((n,n));const=pen*len(req)
    for p in req:
        members=[i for i,r in enumerate(routes) if p in r.points]
        for i in members:lin[i]-=pen
        for i,j in itertools.combinations(members,2):pair[i,j]+=2*pen
    # coverage E = pen*(Σxi - 1)^2 per point, with xi²=xi; diag should be -pen, yes.
    for i,j in itertools.combinations(range(n),2):
        if routes[i].slot==routes[j].slot:pair[i,j]+=pen
    # Normalize all coefficients to control huge phase angles while preserving ordering.
    scale=max(1.,float(np.max(np.abs(lin))),float(np.max(np.abs(pair))))
    return {'routes':routes,'required':req,'n':n,'c':const/scale,'linear':lin/scale,
            'pairs':pair/scale,'scale':scale,'penalty':pen}


def all_bits(n:int)->np.ndarray:
    return ((np.arange(2**n,dtype=np.int64)[:,None]>>np.arange(n,dtype=np.int64))&1).astype(np.uint8)


def qubo_energy(q:dict,bits:np.ndarray)->np.ndarray:
    B=np.atleast_2d(bits).astype(float)
    return q['c']+B@q['linear']+np.einsum('bi,ij,bj->b',B,q['pairs'],B)


def qubo_decode(q:dict,bitvec:np.ndarray)->dict:
    picked=[r for i,r in enumerate(q['routes']) if bitvec[i]==1]
    from collections import Counter
    cnt=Counter(p for r in picked for p in r.points)
    possible=(all(cnt[p]==1 for p in q['required']) and len({r.slot for r in picked})==len(picked))
    return {'ok':possible,'routes':picked,'obj':sum(r.cost for r in picked) if possible else None}


def qaoa_probs(energy:np.ndarray,params:np.ndarray,n:int,p:int=1)->np.ndarray:
    """Ideal QAOA, analytical statevector simulation, same cost and mixer as IBM circuit."""
    state=np.full(2**n,1/np.sqrt(2**n),dtype=complex)
    pairs=np.arange(2**n)
    for layer in range(p):
        g,b=params[2*layer:2*layer+2]
        state*=np.exp(-1j*g*energy)
        cc=np.cos(b);ss=-1j*np.sin(b)
        for bit in range(n):
            zero=pairs[(pairs&(1<<bit))==0]
            one=zero+(1<<bit)
            va=state[zero].copy();vb=state[one].copy()
            state[zero]=cc*va+ss*vb;state[one]=ss*va+cc*vb
    return np.maximum(0,np.abs(state)**2)


def cvar(probs:np.ndarray,energy:np.ndarray,alpha:float=.25)->float:
    order=np.argsort(energy);p=probs[order];e=energy[order]
    cum=np.cumsum(p);k=np.searchsorted(cum,alpha)+1;k=min(k,len(p))
    taken=np.copy(p[:k]);taken[-1]-=max(0,cum[k-1]-alpha)
    return float(np.dot(taken,e[:k])/alpha)


def solve_qubo_ideal(q:dict,shots:int=2048,p:int=1,seed:int=SEED,maxiter:int=35)->dict:
    """Train CVaR ideal QAOA, then simulate shots; not IBM hardware."""
    if q['n']>14:raise ValueError('Máximo 14 variables para statevector local')
    t0=time.perf_counter();B=all_bits(q['n']);E=qubo_energy(q,B)
    rng=np.random.default_rng(seed);best=None
    for k in range(2):
        x0=rng.uniform(0,np.pi,2*p)
        opt=minimize(lambda a:cvar(qaoa_probs(E,a,q['n'],p),E),x0,method='COBYLA',
                     options={'maxiter':maxiter,'rhobeg':.6})
        if best is None or opt.fun<best.fun:best=opt
    probs=qaoa_probs(E,best.x,q['n'],p)
    counts=rng.multinomial(shots,probs)
    feasible=[(qubo_decode(q,b),int(ct),i) for i,(b,ct) in enumerate(zip(B,counts)) if ct>0]
    feasible=[(r,ct,i) for r,ct,i in feasible if r['ok']]
    best_raw=min(feasible,key=lambda u:u[0]['obj']) if feasible else None
    true_feasible=[qubo_decode(q,b) for b in B]
    valid=[r for r in true_feasible if r['ok']]
    opt=min((r['obj'] for r in valid),default=None)
    return dict(params=best.x.tolist(),p=p,shots=shots,n=q['n'],cvar=best.fun,
                qaoa_best=best_raw[0]['obj'] if best_raw else None,
                qaoa_routes=[r.id for r in best_raw[0]['routes']] if best_raw else [],
                valid_shots=sum(int(ct) for r,ct,i in feasible),
                pct_valid=100*sum(int(ct) for r,ct,i in feasible)/shots,
                exact_subproblem=opt,elapsed=time.perf_counter()-t0,
                # Only meaningful within the small neighborhood, not global.
                gap_subproblem_pct=(100*(best_raw[0]['obj']-opt)/max(abs(opt),1e-9) if best_raw and opt is not None else None))


def qubo_qiskit_circuit(q:dict,params:list[float],p:int=1):
    """Build same ideal QAOA as qaoa_probs; imports Qiskit only on demand."""
    from qiskit import QuantumCircuit
    n=q['n'];qc=QuantumCircuit(n,n);qc.h(list(range(n)))
    lin=q['linear'];P=q['pairs']
    h=-lin/2-np.sum(P,axis=1)/4-np.sum(P,axis=0)/4
    for layer in range(p):
        gamma,beta=params[2*layer:2*layer+2]
        for j in range(n):
            if abs(h[j])>1e-12:qc.rz(2*gamma*h[j],j)
        for i,j in zip(*np.nonzero(np.triu(P,k=1))):qc.rzz(2*gamma*P[i,j]/4,i,j)
        for j in range(n):qc.rx(2*beta,j)
    qc.measure(range(n),range(n))
    return qc


def ibm_sample(q:dict,params:list[float],backend_name:str|None=None,shots:int=2048,p:int=1,instance_crn:str|None=None)->dict:
    """Opt-in hardware only. Does not store tokens or run without an explicit call.
    Uses 0.50+ client-side Sampler API. Check quota/job priority before starting.
    """
    from getpass import getpass
    from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
    from qiskit.transpiler import generate_preset_pass_manager
    token=getpass('IBM Quantum API token (no se guarda): ')
    kwargs={'token':token}
    if instance_crn:kwargs['instance']=instance_crn
    service=QiskitRuntimeService(**kwargs)
    backend=service.backend(backend_name) if backend_name else service.least_busy(operational=True,simulator=False)
    circuit=qubo_qiskit_circuit(q,params,p)
    pm=generate_preset_pass_manager(backend=backend,optimization_level=3)
    isa=pm.run(circuit)
    sampler=Sampler(mode=backend)
    job=sampler.run([isa],shots=shots)
    result=job.result()[0]
    # For full classical-bit register the conventional key is 'c'.
    counts=result.data.c.get_counts()
    valid=0;best=None
    for string,ct in counts.items():
        digits=string.replace(' ','')
        x=np.array([int(z) for z in digits[::-1]],dtype=int)
        decoded=qubo_decode(q,x)
        if decoded['ok']:
            valid+=ct
            if best is None or decoded['obj']<best:best=decoded['obj']
    return dict(backend=backend.name,job_id=job.job_id(),shots=shots,valid=valid,
                pct_valid=100*valid/shots,objective_best=best,circuit_depth=isa.depth(),
                two_qubit_gates=sum(cnt for gate,cnt in isa.count_ops().items() if gate in ('cx','cz','ecr','rzz')))


def export_json(path:str|Path,scenario:Scenario,result:dict,metadata:Optional[dict]=None)->Path:
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    body={'scenario':scenario.name,'data_classification':'synthetic/illustrative',
          'summary':summary(scenario,result),'routes':[asdict(r) for r in result.get('routes',[])],
          'validation':result.get('validation',{}),'metadata':metadata or {}}
    path.write_text(json.dumps(body,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    return path


def uniform_baseline(q:dict,shots:int=2048,seed:int=SEED)->dict:
    """Fair random comparator, identical small QUBO feasible predicate and samples."""
    rng=np.random.default_rng(seed)
    bits=rng.integers(0,2,size=(shots,q['n']),dtype=np.int8)
    result=[qubo_decode(q,b) for b in bits]
    valid=[z for z in result if z['ok']]
    return {'shots':shots,'pct_valid':100*len(valid)/shots,
            'best':min((z['obj'] for z in valid),default=None),
            'pct_exact':None}


def reconstruct_hybrid(s:Scenario,master:dict,neigh:dict,selected_route_ids:list[str])->dict:
    """Overlay a measured feasible neighborhood on the classical feasible plan."""
    original=[r for r in master['routes'] if r.slot not in neigh['slots']]
    look={r.id:r for r in neigh['routes']}
    if any(x not in look for x in selected_route_ids):
        return {'ok':False,'status':'unknown sampled route','routes':[]}
    chosen=original+[look[rid] for rid in selected_route_ids]
    verification=validate(s,chosen)
    return {'ok':verification['ok'],'status':'hybrid_candidate' if verification['ok'] else 'rejected',
            'routes':chosen,'obj':sum(r.cost for r in chosen) if verification['ok'] else None,
            'validation':verification,'delta_vs_master':sum(r.cost for r in chosen)-master['obj'] if verification['ok'] else None}


## 2. Configuración reproducible
Cambia `TAMANO` entre 8, 10, 25 y 50. Por defecto no se utiliza hardware real ni API token.

In [ ]:
from fen_v2 import *
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

TAMANO = 8
SEMILLA = SEED
LIMITE_MILP_SEGUNDOS = 25
SHOTS = 2048
PROFUNDIDAD_QAOA = 1
EJECUTAR_QPU_IBM = False  # Cambiar conscientemente a True para gastar cuota
QPU_NOMBRE = None          # None: QPU operativa menos ocupada; o "ibm_kingston" etc.
CRN_INSTANCIA = None      # Solo si IBM lo solicita
SALIDAS = Path('resultados_fen_v2')
SALIDAS.mkdir(exist_ok=True)
print('Modo seguro: IBM deshabilitado por defecto. Semilla:', SEMILLA)

## 3. Construir escenario y validar supuestos
Para 8 puntos se preservan las ubicaciones ilustrativas, la demanda calculada y la flota del notebook V1. Los tamaños superiores incorporan **nuevos puntos sintéticos**, no albergues validados.

In [ ]:
esc = make_scenario(TAMANO, SEMILLA)
print('Instancia:', esc.name)
print('Puntos:',len(esc.stops),'Vehículos:',len(esc.vehicles),'Slots de viajes:',len(esc.slots))
print('Demanda:',sum(s.demand for s in esc.stops.values()),'paletas')
print('Factores de desvío:',esc.detours)
pd.DataFrame([{'id':p.id,'lugar':p.region,'demanda':p.demand,'prioridad':p.priority,
               'desde':clock(p.start),'hasta':clock(p.end),'inundado':p.flooded,'origen':p.source}
              for p in esc.stops.values()]).head(15)

## 4. Baseline exacto sobre el conjunto de rutas
**Selección MILP:** cada punto una sola vez; cada slot vehículo-viaje admite a lo sumo una ruta. Para 8 puntos el pool es exhaustivo bajo las restricciones de esta V2. Para 10/25/50 usa rutas generadas heurísticamente y la optimalidad, cuando existe, es *solo respecto al pool*.

In [ ]:
t0=time.perf_counter()
pool=route_pool(esc,SEMILLA)
master=solve_master(esc,pool,time_limit=LIMITE_MILP_SEGUNDOS)
print('Rutas candidatas:',len(pool))
print('Estado:',master['status'],'| Solución factible:',master['ok'])
print('Certificado óptimo del pool:',master.get('optimal',False),'| Segundos:',round(master['elapsed'],3))
if not master['ok']:
    raise RuntimeError('No se encontró una solución: revisar ventanas, capacidad, slots y pool.')
print('Validación independiente:',validate(esc,master['routes']))
print('Resumen:',summary(esc,master))
for r in master['routes']:
    print(f'{r.slot:8s} D0 -> '+ ' -> '.join(r.points) + f' -> D0 | {r.load} pal. | {r.drive:.1f} min en marcha | regreso {clock(r.return_at)}')
export_json(SALIDAS/f'plan_{TAMANO}.json',esc,master,metadata={'semilla':SEMILLA,'metodo':'MILP_HiGHS','tipo':'instancia_sintetica'})

## 5. Visualizar el plan
**Atención:** los trazos son líneas entre coordenadas, **NO rutas viales de OpenStreetMap**. Solo sirven para inspeccionar asignaciones espaciales.

In [ ]:
fig,ax=plt.subplots(figsize=(8,7))
for p in esc.stops.values():
    ax.scatter(p.lon,p.lat,s=50 if p.priority==3 else 30,marker='^' if p.flooded else 'o',color='#F97316' if p.priority==3 else '#2563EB')
    ax.annotate(p.id,(p.lon,p.lat),xytext=(4,4),textcoords='offset points',fontsize=8)
ax.scatter(DEPOT[2],DEPOT[1],marker='s',s=130,color='#111827',label='Depósito ficticio')
colors=plt.cm.tab20.colors
for idx,r in enumerate(master['routes']):
    seq=['D0']+list(r.points)+['D0']
    ax.plot([coords(esc,k)[1] for k in seq],[coords(esc,k)[0] for k in seq],
            label=r.slot,color=colors[idx%len(colors)],linewidth=1.8,alpha=.8)
ax.set(title='Bajo Piura - solución MILP (geometría ilustrativa)',xlabel='Longitud',ylabel='Latitud')
ax.legend(fontsize=7,ncol=2);ax.grid(alpha=.15)
plt.tight_layout();plt.show()

## 6. Subproblema QUBO + QAOA ideal
Se identifican **dos slots con alternativas factibles** y se deja fija la parte restante del plan clásico. El QUBO penaliza la falta/sobrecobertura de cada punto y que un mismo slot seleccione más de una ruta.

- **Óptimo de referencia:** enumeración exacta de todas las cadenas del QUBO pequeño.
- **Baseline aleatorio:** muestreo uniforme sobre el mismo QUBO.
- **Cuántico ideal:** QAOA de profundidad configurable, CVaR y 2 reinicios; **sin ruido IBM**.
- **Plan híbrido:** sustituye las dos rutas del plan MILP por una combinación que QAOA haya muestreado y que pase la validación.

La solución QAOA no puede mejorar el óptimo certificado del MILP sobre el mismo pool; su valor experimental es evaluar factibilidad, brecha y recursos, no alegar superioridad.

In [ ]:
vec=select_neighborhood(esc,master,pool,max_variables=10)
qubo=make_qubo(vec)
print('Slots cuánticos:',sorted(vec['slots']),'| Cobertura local:',sorted(vec['required']))
print('Variables binarias:',qubo['n'],'| Pares factibles disponibles:',vec['complete_pairs_available'])
ideal=solve_qubo_ideal(qubo,shots=SHOTS,p=PROFUNDIDAD_QAOA,seed=SEMILLA,maxiter=70)
azar=uniform_baseline(qubo,shots=SHOTS,seed=SEMILLA)
comparacion=pd.DataFrame([
 {'Método':'Enumeración exacta (sub-QUBO)','Objetivo':ideal['exact_subproblem'],'Factibles (%)':100.0},
 {'Método':'Muestreo uniforme','Objetivo':azar['best'],'Factibles (%)':azar['pct_valid']},
 {'Método':'QAOA ideal (NO IBM)','Objetivo':ideal['qaoa_best'],'Factibles (%)':ideal['pct_valid']}])
display(comparacion)
print('QAOA: brecha sub-QUBO (%)',ideal['gap_subproblem_pct'], '| Segundos',round(ideal['elapsed'],3))
plan_hibrido=reconstruct_hybrid(esc,master,vec,ideal['qaoa_routes'])
print('Plan híbrido reconstruido:',plan_hibrido['status'],'| objetivo:',plan_hibrido.get('obj'))
print('Diferencia contra MILP (misma función):',plan_hibrido.get('delta_vs_master'))

## 7. Replanificación antes del despacho
Se modifica el tiempo de desvío de dos conexiones del grafo abstracto. **No se simula aquí la eliminación física de aristas viales ni la posición de vehículos en tránsito.** El siguiente módulo GIS deberá reemplazar esos factores por distancias/tiempos extraídos de una red vial real.

In [ ]:
evento=make_scenario(TAMANO,SEMILLA,detour_override={('P7','P8'):6.0,('D0','P8'):4.0})
pool_evento=route_pool(evento,SEMILLA)
plan_evento=solve_master(evento,pool_evento,time_limit=LIMITE_MILP_SEGUNDOS)
if not plan_evento['ok']:
    print('No hay solución factible para este evento; documentar el fallo y ajustar parámetros.')
else:
    tabla=pd.DataFrame([
     {'Escenario':'Inicial','Objetivo':master['obj'],'Viaje (min)':summary(esc,master)['travel_minutes'],'Factible':master['ok']},
     {'Escenario':'Desvíos incrementados','Objetivo':plan_evento['obj'],'Viaje (min)':summary(evento,plan_evento)['travel_minutes'],'Factible':plan_evento['ok']}
    ])
    display(tabla)
    export_json(SALIDAS/f'replan_{TAMANO}.json',evento,plan_evento,metadata={'tipo':'evento_sintetico_pre_despacho'})
    print('Nuevas rutas:', [(r.slot,r.points) for r in plan_evento['routes']])

## 8. Benchmark de escalabilidad
Puede tomar algo de tiempo para la instancia 50. Los resultados dependen del generador de rutas, del límite de resolución y de esta máquina. El benchmark de PyVRP/OR-Tools aún debe añadirse en el siguiente sprint.

In [ ]:
bench=[]
for tam in (8,10,25,50):
    s=make_scenario(tam,SEMILLA)
    pool_s=route_pool(s,SEMILLA)
    res_s=solve_master(s,pool_s,time_limit=20 if tam==50 else 12)
    rec=summary(s,res_s)
    rec['pool_size']=len(pool_s)
    rec['escenario']=tam
    bench.append(rec)
df_benchmark=pd.DataFrame(bench)
display(df_benchmark[['escenario','vehicles','trip_slots','pool_size','ok','optimal','elapsed','obj','travel_minutes']])
df_benchmark.to_csv(SALIDAS/'benchmark_sintetico.csv',index=False)
print('NOTA: optimal=True certifica el óptimo respecto al pool, no del VRPTW irrestricto en 10/25/50.')

## 9. Ejecución opcional en IBM Quantum

**Por defecto DESACTIVADA.** Antes de usar: instala `qiskit` y `qiskit-ibm-runtime` compatibles; confirma cuota, CRN y QPU disponible. No compartas tokens. Se ejecuta únicamente el subproblema `qubo`, nunca el VRPTW de 50 puntos entero.

En Colab ejecuta primero: `%pip install -q "qiskit>=2.5,<3" "qiskit-ibm-runtime>=0.50,<1"` y reinicia el runtime si lo solicita. Si actualizas la librería, revisa compatibilidad con la documentación oficial del Sampler.

In [ ]:
if EJECUTAR_QPU_IBM:
    if qubo['n']>14:
        raise ValueError('Reducir el QUBO antes de enviar a hardware.')
    print('ADVERTENCIA: va a consumir cuota de IBM Quantum.')
    dato_IBM=ibm_sample(qubo,ideal['params'],backend_name=QPU_NOMBRE,
                        shots=SHOTS,p=PROFUNDIDAD_QAOA,instance_crn=CRN_INSTANCIA)
    print('Evidencia IBM:',dato_IBM)
    (SALIDAS/'evidencia_ibm.json').write_text(json.dumps(dato_IBM,indent=2),encoding='utf-8')
else:
    print('No se ejecutó hardware IBM; todos los resultados anteriores son clásicos/ideal-simulado.')

## 10. Exportación y controles finales

Comparte el JSON y el CSV con el equipo de producto. Los archivos exportados no incluyen API tokens. Los registros en QPU real incluyen *job ID* y backend solo cuando se autorice su ejecución. Los ensayos de 25/50 son escenarios ficticios generados y no son evidencia de factibilidad operativa en Piura.

In [ ]:
import platform, sys
metadata={'version':'FEN-Piura-V2.0','seed':SEMILLA,'python':platform.python_version(),
          'numpy':np.__version__, 'mip':'SciPy/HiGHS',
          'qaoa_ideal':ideal,'qaoa_uniforme':azar,
          'fuentes':'Escenario ficticio heredado de V1, sin red vial real'}
# Metadata is JSON-friendly, no hidden API key and no secrets.
export_json(SALIDAS/f'proyecto_resumen_{TAMANO}.json',esc,master,metadata)
print('Carpeta de resultados:',SALIDAS.resolve())
print('Archivos:',*[p.name for p in SALIDAS.iterdir()],sep=' / ')
print('AVISO: no utilizar estas rutas para distribución real.')

## Trabajo pendiente asignable al equipo

**Datos/GIS:** fuente de albergues, inventario, demanda, geometría de red vial y atributos de vehículos.  
**Clásico:** PyVRP/OR-Tools con igual función objetivo y ventanas, benchmark, múltiples viajes sin turnos rígidos.  
**Cuántico:** incorporar warm-start, pruebas p=1/2, repeticiones, errores de puerta y dos QPUs si hay cuota.  
**Producto:** mapa con calles verdaderas, selector de escenario, tablero, reporte de reproducibilidad.  
**Validación:** comparar resultados con el mismo validador y guardar el registro de cada corrida. 

Consulta `Documentacion_Proyecto_FEN_Piura_V2.pdf` para matriz RACI, fases y criterios de aceptación.